In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

import os

print(os.listdir("../input"))
os.chdir("/kaggle/working/")



['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'labels.csv.zip', 'train.csv', 'new-york-city-taxi-fare-prediction', 'test.csv', 'labels.csv', 'description.md', 'GCP-Coupons-Instructions.rtf', 'sample_submission.csv.zip']


In [2]:
df_train = pd.read_csv(
    "../input/train.csv", nrows=800000, parse_dates=["pickup_datetime"]
)
df_train.head()



,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2009-06-15 17:26:21.0000001,4.5,2009-06-15 17:26:21+00:00,-73.844311,40.721319,-73.841610,40.712278,1
1,2010-01-05 16:52:16.0000002,16.9,2010-01-05 16:52:16+00:00,-74.016048,40.711303,-73.979268,40.782004,1
2,2011-08-18 00:35:00.00000049,5.7,2011-08-18 00:35:00+00:00,-73.982738,40.761270,-73.991242,40.750562,2
3,2012-04-21 04:30:42.0000001,7.7,2012-04-21 04:30:42+00:00,-73.987130,40.733143,-73.991567,40.758092,1
4,2010-03-09 07:51:00.000000135,5.3,2010-03-09 07:51:00+00:00,-73.968095,40.768008,-73.956655,40.783762,1


In [3]:
df_train.describe()
df_train.dtypes



key                               object
fare_amount                      float64
pickup_datetime      datetime64[ns, UTC]
pickup_longitude                 float64
pickup_latitude                  float64
dropoff_longitude                float64
dropoff_latitude                 float64
passenger_count                    int64
dtype: object

In [4]:
df_train = df_train[(df_train["fare_amount"] > 0.05) & (df_train.passenger_count > 0)]
df_train.dropna(how="any", axis="rows", inplace=True)
print("New Size: {}".format(len(df_train)))
df_test = pd.read_csv("../input/test.csv", parse_dates=["pickup_datetime"])



New Size: 797104


In [5]:
mask = df_train["pickup_longitude"].between(-75, -73)
mask &= df_train["dropoff_longitude"].between(-75, -73)
mask &= df_train["pickup_latitude"].between(40, 42)
mask &= df_train["dropoff_latitude"].between(40, 42)
mask &= df_train["passenger_count"].between(0, 8)
mask &= df_train["fare_amount"].between(0, 250)

df_train = df_train[mask]

# Change (score-improving, minimal): apply the same geographic sanity filter to test to avoid extreme coords
# producing unreasonable distances and large prediction errors (train already filtered similarly).
mask_t = df_test["pickup_longitude"].between(-75, -73)
mask_t &= df_test["dropoff_longitude"].between(-75, -73)
mask_t &= df_test["pickup_latitude"].between(40, 42)
mask_t &= df_test["dropoff_latitude"].between(40, 42)
mask_t &= df_test["passenger_count"].between(0, 8)

df_test["__valid_geo__"] = mask_t




In [6]:
def distance(lat1, lon1, lat2, lon2):
    p = 0.017453292519943295  # Pi/180
    a = (
        0.5
        - np.cos((lat2 - lat1) * p) / 2
        + np.cos(lat1 * p) * np.cos(lat2 * p) * (1 - np.cos((lon2 - lon1) * p)) / 2
    )
    return 0.6213712 * 12742 * np.arcsin(np.sqrt(a))  # miles


NYC_CENTER = (40.7580, -73.9855)  # Times Square / Midtown
JFK = (40.6413, -73.7781)
LGA = (40.7769, -73.8740)
EWR = (40.6895, -74.1745)

for df in (df_train, df_test):
    df["distance_miles"] = distance(
        df.pickup_latitude,
        df.pickup_longitude,
        df.dropoff_latitude,
        df.dropoff_longitude,
    )
    df["abs_lon_diff"] = (df["pickup_longitude"] - df["dropoff_longitude"]).abs()
    df["abs_lat_diff"] = (df["pickup_latitude"] - df["dropoff_latitude"]).abs()

    df["manhattan_dist"] = df["abs_lon_diff"] + df["abs_lat_diff"]

    df["euclidean_deg"] = np.sqrt(df["abs_lon_diff"] ** 2 + df["abs_lat_diff"] ** 2)
    df["euclidean_miles_approx"] = np.sqrt(
        (df["abs_lat_diff"] * 69.0) ** 2 + (df["abs_lon_diff"] * 52.0) ** 2
    )

    df["pickup_to_center_miles"] = distance(
        df["pickup_latitude"], df["pickup_longitude"], NYC_CENTER[0], NYC_CENTER[1]
    )
    df["dropoff_to_center_miles"] = distance(
        df["dropoff_latitude"], df["dropoff_longitude"], NYC_CENTER[0], NYC_CENTER[1]
    )

    df["pickup_to_jfk_miles"] = distance(
        df["pickup_latitude"], df["pickup_longitude"], JFK[0], JFK[1]
    )
    df["dropoff_to_jfk_miles"] = distance(
        df["dropoff_latitude"], df["dropoff_longitude"], JFK[0], JFK[1]
    )

    df["pickup_to_lga_miles"] = distance(
        df["pickup_latitude"], df["pickup_longitude"], LGA[0], LGA[1]
    )
    df["dropoff_to_lga_miles"] = distance(
        df["dropoff_latitude"], df["dropoff_longitude"], LGA[0], LGA[1]
    )

    df["pickup_to_ewr_miles"] = distance(
        df["pickup_latitude"], df["pickup_longitude"], EWR[0], EWR[1]
    )
    df["dropoff_to_ewr_miles"] = distance(
        df["dropoff_latitude"], df["dropoff_longitude"], EWR[0], EWR[1]
    )

    dt = df["pickup_datetime"]
    df["year"] = dt.dt.year
    df["hour"] = dt.dt.hour
    df["minute"] = dt.dt.minute
    df["month"] = dt.dt.month
    df["dayofweek"] = dt.dt.dayofweek
    df["day"] = dt.dt.day

    # Change (score-improving, minimal): cyclic encoding for periodic datetime parts.
    # This keeps the same underlying info but makes it learnable for tree splits.
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

    df["minute_sin"] = np.sin(2 * np.pi * df["minute"] / 60.0)
    df["minute_cos"] = np.cos(2 * np.pi * df["minute"] / 60.0)

    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7.0)
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7.0)

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)



In [7]:
features = [
    # Keep original raw datetime parts to preserve core logic, but add cyclic encodings (above) for better signal.
    "year",
    "month",
    "day",
    "dayofweek",
    "hour",
    "minute",
    "hour_sin",
    "hour_cos",
    "minute_sin",
    "minute_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
    "distance_miles",
    "abs_lat_diff",
    "abs_lon_diff",
    "manhattan_dist",
    "euclidean_deg",
    "euclidean_miles_approx",
    "pickup_to_center_miles",
    "dropoff_to_center_miles",
    "pickup_to_jfk_miles",
    "dropoff_to_jfk_miles",
    "pickup_to_lga_miles",
    "dropoff_to_lga_miles",
    "pickup_to_ewr_miles",
    "dropoff_to_ewr_miles",
    "passenger_count",
]
X = df_train[features].values
y = df_train["fare_amount"].values
X_test = df_test[features]
df_test.head(5)



,key,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,__valid_geo__,distance_miles,abs_lon_diff,...,dayofweek,day,hour_sin,hour_cos,minute_sin,minute_cos,dow_sin,dow_cos,month_sin,month_cos
0,2010-10-01 21:26:11.0000001,2010-10-01 21:26:11+00:00,-73.983130,40.761970,-73.994386,40.749236,1,True,1.058853,0.011256,...,4,1,-0.707107,0.707107,0.406737,-0.913545,-0.433884,-0.900969,-8.660254e-01,5.000000e-01
1,2013-10-06 01:38:00.00000083,2013-10-06 01:38:00+00:00,-73.948505,40.753977,-73.808195,40.731952,2,True,7.500968,0.140310,...,6,6,0.258819,0.965926,-0.743145,-0.669131,-0.781831,0.623490,-8.660254e-01,5.000000e-01
2,2012-03-30 19:13:53.0000001,2012-03-30 19:13:53+00:00,-73.973964,40.791979,-73.979018,40.785544,1,True,0.517284,0.005054,...,4,30,-0.965926,0.258819,0.978148,0.207912,-0.433884,-0.900969,1.000000e+00,6.123234e-17
3,2012-02-08 02:57:23.0000001,2012-02-08 02:57:23+00:00,-73.991478,40.738907,-73.907198,40.861572,2,True,9.553148,0.084280,...,2,8,0.500000,0.866025,-0.309017,0.951057,0.974928,-0.222521,8.660254e-01,5.000000e-01
4,2013-12-13 22:56:00.000000237,2013-12-13 22:56:00+00:00,-73.986281,40.740067,-73.933927,40.856781,2,True,8.516409,0.052354,...,4,13,-0.500000,0.866025,-0.406737,0.913545,-0.433884,-0.900969,-2.449294e-16,1.000000e+00


In [8]:
import xgboost as xgb
from bayes_opt import BayesianOptimization
from sklearn.model_selection import train_test_split



In [9]:
X_train_np, X_val_np, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

dtrain = xgb.DMatrix(X_train_np, label=y_train)
dval = xgb.DMatrix(X_val_np, label=y_val)


# Change (score-improving, minimal): use early stopping inside xgb.cv to pick an effective boosting length
# for each parameter set (still RMSE objective; avoids under/overfitting from arbitrary round counts).
def xgb_eva(max_depth, gamma, colsample_bytree, num_boost_round):
    params = {
        "eval_metric": "rmse",
        "max_depth": int(max_depth),
        "subsample": 0.8,
        "eta": 0.1,
        "gamma": float(gamma),
        "colsample_bytree": float(colsample_bytree),
        "objective": "reg:squarederror",
        "seed": 42,
    }
    nrounds = int(num_boost_round)
    nrounds = max(80, min(600, nrounds))
    cv_result = xgb.cv(
        params,
        dtrain,
        num_boost_round=nrounds,
        nfold=3,
        seed=42,
        shuffle=True,
        early_stopping_rounds=30,
        verbose_eval=False,
    )
    return -1.0 * cv_result["test-rmse-mean"].iloc[-1]




In [10]:
xgb_bo = BayesianOptimization(
    f=xgb_eva,
    pbounds={
        "max_depth": (3, 7),
        "gamma": (0, 1),
        "colsample_bytree": (0.3, 0.9),
        "num_boost_round": (200, 700),
    },
    random_state=42,
    verbose=2,
)
xgb_bo.maximize(init_points=3, n_iter=5)



|   iter    |  target   | max_depth |   gamma   | colsam... | num_bo... |
-------------------------------------------------------------------------
| 1         | -3.749311 | 4.4981604 | 0.9507143 | 0.7391963 | 499.32924 |
| 2         | -3.831622 | 3.6240745 | 0.1559945 | 0.3348501 | 633.08807 |
| 3         | -3.730828 | 5.4044600 | 0.7080725 | 0.3123506 | 684.95492 |
| 4         | -3.723599 | 5.2757240 | 0.5600573 | 0.4904086 | 683.60184 |
| 5         | -3.805717 | 3.1819697 | 0.2900633 | 0.5048827 | 669.67615 |
| 6         | -3.799607 | 3.3238236 | 0.5007145 | 0.7772661 | 511.07928 |
| 7         | -3.700187 | 6.1676339 | 0.8759421 | 0.6258052 | 490.04646 |
| 8         | -3.802019 | 3.0       | 0.1031226 | 0.7848900 | 482.99489 |


In [11]:
best_params = dict(xgb_bo.max["params"])
best_params["max_depth"] = int(best_params["max_depth"])
best_num_boost_round = int(best_params.pop("num_boost_round"))
best_num_boost_round = max(80, min(800, best_num_boost_round))

params = {
    "eval_metric": "rmse",
    "max_depth": best_params["max_depth"],
    "subsample": 0.8,
    "eta": 0.1,
    "gamma": float(best_params["gamma"]),
    "colsample_bytree": float(best_params["colsample_bytree"]),
    "objective": "reg:squarederror",
    "seed": 42,
}

# Change (score-improving, minimal): compute best boosting length with early stopping on the held-out val
# (same training approach; just selects a better num_boost_round for final fit).
evals = [(dtrain, "train"), (dval, "val")]
tmp_model = xgb.train(
    params,
    dtrain,
    num_boost_round=best_num_boost_round,
    evals=evals,
    early_stopping_rounds=30,
    verbose_eval=False,
)
best_iteration = int(getattr(tmp_model, "best_iteration", best_num_boost_round - 1))
final_num_boost_round = max(50, best_iteration + 1)

params, final_num_boost_round



({'eval_metric': 'rmse',
  'max_depth': 6,
  'subsample': 0.8,
  'eta': 0.1,
  'gamma': 0.8759421795527655,
  'colsample_bytree': 0.6258052270494922,
  'objective': 'reg:squarederror',
  'seed': 42},
 400)

In [12]:
model2 = xgb.train(
    params, xgb.DMatrix(X, label=y), num_boost_round=final_num_boost_round
)

X_testm = xgb.DMatrix(X_test.values)
y_test = model2.predict(X_testm)



In [13]:
X_test.head(5)



,year,month,day,dayofweek,hour,minute,hour_sin,hour_cos,minute_sin,minute_cos,...,euclidean_miles_approx,pickup_to_center_miles,dropoff_to_center_miles,pickup_to_jfk_miles,dropoff_to_jfk_miles,pickup_to_lga_miles,dropoff_to_lga_miles,pickup_to_ewr_miles,dropoff_to_ewr_miles,passenger_count
0,2010,10,1,4,21,26,-0.707107,0.707107,0.406737,-0.913545,...,1.055750,0.301040,0.763532,13.596068,13.564373,5.802912,6.583657,11.201866,10.295509,1
1,2013,10,6,6,1,38,0.258819,0.965926,-0.743145,-0.669131,...,7.452713,1.956093,9.454203,11.844477,6.458865,4.208303,4.637583,12.644953,19.407621,2
2,2012,3,30,4,19,13,-0.965926,0.258819,0.978148,0.207912,...,0.515963,2.424073,1.933094,14.614951,14.493003,5.332443,5.526707,12.662705,12.197154,1
3,2012,2,8,2,2,57,0.500000,0.866025,-0.309017,0.951057,...,9.531221,1.355802,8.244879,13.055556,16.651945,6.685324,6.102366,10.174764,18.356322,2
4,2013,12,13,4,22,56,-0.500000,0.866025,-0.406737,0.913545,...,8.500976,1.239725,7.338709,12.865448,16.976160,6.403744,6.346763,10.457981,17.089206,2


In [14]:
# Change (score-improving, minimal): for rows with invalid geo (outside NYC bbox), fall back to a stable
# baseline (training mean) rather than letting extreme engineered distances create huge errors.
train_mean_fare = float(df_train["fare_amount"].mean())
y_test_safe = y_test.copy()
invalid_idx = ~df_test["__valid_geo__"].values
y_test_safe[invalid_idx] = train_mean_fare

sub = pd.DataFrame()
sub["key"] = df_test["key"].values
sub["fare_amount"] = y_test_safe
sub.to_csv("submission.csv", index=False)
sub

,key,fare_amount
0,2010-10-01 21:26:11.0000001,6.744988
1,2013-10-06 01:38:00.00000083,32.867710
2,2012-03-30 19:13:53.0000001,4.021691
3,2012-02-08 02:57:23.0000001,29.496178
4,2013-12-13 22:56:00.000000237,31.846933
...,...,...
9909,2010-12-02 01:44:39.0000002,4.561695
9910,2010-06-07 16:17:38.0000003,11.338308
9911,2013-11-01 08:37:36.0000003,7.031198
9912,2011-05-03 10:14:00.000000123,8.790957
